# Exercise 1: A dated resource audit for five languages

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Endangered Languages and Ethics · version 0.1.0*

> Audit five languages with explicit language codes for every source and a dated UD release. Show not-found, error and not-checked values distinctly, and explain why repository-count ratios cannot be converted into person-hours of development.

**Setup.** `pip install requests huggingface_hub fasttext-wheel transformers torch sentencepiece "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this, and Exercise 1 also calls `git`. Data come from commit- or tag-pinned GitHub files and are hash-checked: the UD 2.17 TueCL treebanks, the KazNERD and KyrgyzNER releases and their README files, and the TIL corpus README. The audit in Exercise 1 also reads the book's UD 2.17 inventory (`tables/tbl-ud217-inventory/data.csv`), the NLLB-200 tokeniser (`NLLB_PATH`) and GlotLID (`GLOTLID_PATH`), and queries live services, so its live columns are dated observations, not pinned results. Run the notebooks from the `exercises/` folder: the first code cell imports the chapter's `observation` and `hub_count` helpers from `../code/remaining/resources.py`. Downloads are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`).

The chapter's `observation` and `hub_count` helpers record every cell with its source, query, status and time. Columns: UD 2.17 (the book's inventory table), the Apertium monolingual repository (`git ls-remote`), the TIL corpus language-pair table (pinned README), X-WMT test files and SIB-200 (pinned GitHub files), NLLB-200 language codes, GlotLID labels and Hugging Face Hub model tags. Common Voice is deliberately left *not checked*.

In [ ]:
import collections, csv, datetime, difflib, hashlib, json, os, re, statistics, subprocess, sys, time
from pathlib import Path
import requests
sys.path.insert(0, str(Path("../code/remaining").resolve()))
from resources import observation, hub_count

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/"
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
UD = "https://raw.githubusercontent.com/UniversalDependencies/{}/r2.17/{}"
SHA256 = {
    KAZNERD + "README.md": "2d56859aa35eadafb8bc560b375d5128627e3bab3517bc74377c7818905e3614",
    KAZNERD + "KazNERD/IOB2_train.txt": "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD + "KazNERD/IOB2_test.txt": "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER + "README.md": "c5beec0dd94156f5cdae4c5f7ffafeb23704f6fd1024ed4cb505cc8ef498a46c",
    TIL + "til_corpus/README.md": "da88f81cc7a23fb46425dcf236f7c89ad0f63663d2141d8fb7165ea8015308d8",
    UD.format("UD_Turkish-TueCL", "tr_tuecl-ud-test.conllu"): "2508f47e57b9b6a1d7241d0ab4a918d84c5cbb0455bbd961ac5231e31a1d311e",
    UD.format("UD_Azerbaijani-TueCL", "az_tuecl-ud-test.conllu"): "8b315f8b065f094e38482465706ba44a9caf1b082fc8bbc94908f6fdeee8c3b5",
    UD.format("UD_Uzbek-TueCL", "uz_tuecl-ud-test.conllu"): "23257711196df0bbfaf5b25397d7c6102fbc8800ed83b15eb77342929ee0e469",
    UD.format("UD_Kyrgyz-TueCL", "ky_tuecl-ud-test.conllu"): "08224be6b74346f582234f54527412dd6265c8d8b42e7d6ef463aade85bdcd9b",
}
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256(path.read_bytes()).hexdigest(); FETCHED[url] = h
    if url in SHA256 and h != SHA256[url]:
        raise ValueError(f"{url} differs from the pinned file")
    return path

def read_conll(path):
    """KazNERD-style column file -> [(words, tags)]."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith("# "):
            c = line.split("\t") if "\t" in line else line.split()
            words.append(c[0]); tags.append(c[-1].strip())
    return sents

def show(obs):
    """One audit cell: the value, or the reason why there is none. Zero, not found, error and not checked stay distinct."""
    return {"observed": lambda o: str(o["value"]), "not_found": lambda o: "not found", "error": lambda o: "ERROR",
            "not_checked": lambda o: "not checked"}[obs["status"]](obs)
print("helpers loaded from code/remaining/resources.py:", observation.__name__, hub_count.__name__)

In [ ]:
# Five languages with very different resource situations. Codes are given explicitly for every source.
LANGS = {  # name: (ISO 639-3, UD name, Apertium repo, TIL code, FLORES/NLLB/GlotLID code, Hub tag aliases)
    "Kazakh": ("kaz", "Kazakh", "apertium-kaz", "kk", "kaz_Cyrl", ["kk", "kaz"]),
    "Kyrgyz": ("kir", "Kyrgyz", "apertium-kir", "ky", "kir_Cyrl", ["ky", "kir"]),
    "Nogai": ("nog", "Nogai", "apertium-nog", None, "nog_Cyrl", ["nog"]),
    "Tuvan": ("tyv", "Tuvan", "apertium-tyv", "tyv", "tyv_Cyrl", ["tyv"]),
    "Shor": ("cjs", "Shor", "apertium-cjs", "cjs", "cjs_Cyrl", ["cjs"]),
}
inventory = list(csv.DictReader(open("../tables/tbl-ud217-inventory/data.csv", encoding="utf-8")))
def ud(name):
    rows = [r for r in inventory if r["treebank"].split("-")[0] == name]
    if not rows:
        return observation(0, "observed", "UD 2.17 inventory (tables/tbl-ud217-inventory)", name)
    return observation(f"{len(rows)} tb, {sum(int(r['words']) for r in rows):,} w", "observed", "UD 2.17 inventory", name,
                       ids=[r["treebank"] for r in rows])
def apertium(repo):
    q = f"https://github.com/apertium/{repo}"
    try:
        r = subprocess.run(["git", "ls-remote", q, "HEAD"], capture_output=True, text=True, timeout=60,
                           env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    except Exception as e:
        return observation(None, "error", "git ls-remote", q, error=str(e))
    if r.returncode == 0 and r.stdout.strip():
        return observation("repo " + r.stdout.split()[0][:7], "observed", "git ls-remote", q)
    if "could not read Username" in r.stderr or "not found" in r.stderr:     # GitHub answers a missing public repo this way
        return observation(None, "not_found", "git ls-remote", q, error=r.stderr.strip()[-80:])
    return observation(None, "error", "git ls-remote", q, error=r.stderr.strip()[-200:])
# TIL corpus: training sizes from the language-pair table in the pinned TIL corpus README.
til_rows = []
for line in fetch(TIL + "til_corpus/README.md").read_text(encoding="utf-8").split("## Language pairs", 1)[1].splitlines()[1:]:
    c = [x.strip() for x in line.strip().strip("|").split("|")]
    if c[0].isdigit(): til_rows.append((c[3], c[4], int(c[5])))
    elif til_rows and not line.startswith("|"): break
def til(code):
    if code is None:
        return observation(None, "not_found", "TIL corpus README (pinned)", "no TIL language code")
    rows = [r for r in til_rows if r[0] == code]
    if not rows:
        return observation(None, "not_found", "TIL corpus README (pinned)", code)
    best = max(rows, key=lambda r: r[2])
    return observation(f"{len(rows)} pairs, max {best[2]:,} ({code}-{best[1]})", "observed", "TIL corpus README (pinned)", code)
def url_exists(url, source):
    try:
        r = requests.head(url, timeout=60, allow_redirects=True)
    except requests.RequestException as e:
        return observation(None, "error", source, url, error=str(e)[:200])
    if r.status_code == 200: return observation("yes", "observed", source, url)
    if r.status_code == 404: return observation(None, "not_found", source, url)
    return observation(None, "error", source, url, error=f"HTTP {r.status_code}")
from transformers import AutoTokenizer
nllb_tok = AutoTokenizer.from_pretrained(os.environ.get("NLLB_PATH", str(Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser())))
import fasttext
glot_labels = set(fasttext.load_model(os.environ["GLOTLID_PATH"]).get_labels())
def label_in(code, labels, source):
    return observation("yes", "observed", source, code) if code in labels else observation(None, "not_found", source, code)
try:
    from huggingface_hub import HfApi
    api = HfApi()
except ImportError as e:
    api = None
audit = {}
for name, (iso, udname, repo, til_code, flores, aliases) in LANGS.items():
    audit[name] = {
        "UD 2.17": ud(udname),
        "Apertium": apertium(repo),
        "TIL corpus": til(til_code),
        "X-WMT test": url_exists(f"{TIL}xwmt/test/{til_code}-en/{til_code}-en.{til_code}", "til-mt xwmt/test") if til_code
                      else observation(None, "not_found", "til-mt xwmt/test", "no TIL code"),
        "SIB-200": url_exists(f"{SIB}{flores}/dev.tsv", "SIB-200 (pinned)"),
        "NLLB-200": label_in(flores, set(nllb_tok.get_vocab()), "NLLB-200 tokenizer language codes"),
        "GlotLID": label_in(f"__label__{flores}", glot_labels, "GlotLID v3 labels"),
        "Hub models": hub_count(api, "models", aliases) if api else observation(None, "error", "Hugging Face Hub", aliases, error="huggingface_hub missing"),
        "Common Voice": observation(None, "not_checked", "Common Voice release metadata (not queried here)", iso),
    }
cols = list(next(iter(audit.values())))
print("audit date (UTC):", datetime.datetime.now(datetime.timezone.utc).date())
for name in LANGS:
    print(f"\n{name} ({LANGS[name][0]})")
    for col in cols:
        print(f"  {col:<13} {show(audit[name][col])}")
errors = [(n, c, audit[n][c]["error"]) for n in audit for c in audit[n] if audit[n][c]["status"] == "error"]
print(f"\n{len(errors)} cells are errors, e.g.:", errors[:1])
(CACHE / "ch17-resource-audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=1))
print("full observations (query, status, error, timestamp) written to BOOK_DATA_DIR/ch17-resource-audit.json")

## Worked solution

Output of `experiments/ch17_exercises/run_ch17_exercises.py`:

```
audit date (UTC): 2026-10-20

Kazakh (kaz)
  UD 2.17       1 tb, 10,536 w
  Apertium      repo 95c6dd0
  TIL corpus    19 pairs, max 4,403,385 (kk-ru)
  X-WMT test    yes
  SIB-200       yes
  NLLB-200      yes
  GlotLID       yes
  Hub models    ERROR
  Common Voice  not checked

Kyrgyz (kir)
  UD 2.17       2 tb, 25,378 w
  Apertium      repo c5d15a1
  TIL corpus    20 pairs, max 312,644 (ky-en)
  X-WMT test    yes
  SIB-200       yes
  NLLB-200      yes
  GlotLID       yes
  Hub models    ERROR
  Common Voice  not checked

Nogai (nog)
  UD 2.17       0
  Apertium      repo 09fdef0
  TIL corpus    not found
  X-WMT test    not found
  SIB-200       not found
  NLLB-200      not found
  GlotLID       yes
  Hub models    ERROR
  Common Voice  not checked

Tuvan (tyv)
  UD 2.17       0
  Apertium      repo 038f717
  TIL corpus    10 pairs, max 12,531 (tyv-ba)
  X-WMT test    not found
  SIB-200       not found
  NLLB-200      not found
  GlotLID       yes
  Hub models    ERROR
  Common Voice  not checked

Shor (cjs)
  UD 2.17       0
  Apertium      not found
  TIL corpus    1 pairs, max 2,294 (cjs-ru)
  X-WMT test    not found
  SIB-200       not found
  NLLB-200      not found
  GlotLID       yes
  Hub models    ERROR
  Common Voice  not checked

5 cells are errors, e.g.: [('Kazakh', 'Hub models', '403 Forbidden')]
full observations (query, status, error, timestamp) written to BOOK_DATA_DIR/ch17-resource-audit.json
```

**Keep observation status separate from the value.** The dated audit distinguishes observed counts, entries not found, failed requests and unchecked sources. The five Hub requests returned HTTP 403, so no model counts are available. The absence of Nogai, Tuvan and Shor treebanks is limited to the checked UD 2.17 inventory.

The Nogai Apertium repository lookup succeeds, and GlotLID has a Nogai label. Neither observation tests an analyser or classifier's quality. The pinned TIL table lists ten Tuvan pairs, with a maximum training size of 12,531 sentences, and one Shor–Russian pair of 2,294. The Shor Apertium lookup returns no repository; that is not an exhaustive search for Shor tools.

Repository counts cannot determine development effort. Copies, adapters and multilingual tags may all enter a count, while the intended task, data permissions and maintenance needs remain unmeasured. Common Voice was not checked. Use this inventory to plan further checks with partners, not to rank communities or estimate person-hours.

## Your turn — extension

Add two more languages (for example Chuvash and Karakalpak) with their own codes for every source, run the notebook where the Hub is reachable, and compare the model counts of the 2-letter and 3-letter tags separately before merging them.